In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "/project/jevans/tzhang3/models/Qwen3-VL-8B-Instruct"  # Or local path

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="auto",
).eval()

print("Qwen3-VL loaded")
print("Class:", model.__class__.__name__)
print("Architectures:", getattr(model.config, "architectures", None))
model

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Qwen3-VL loaded
Class: Qwen3VLForConditionalGeneration
Architectures: ['Qwen3VLForConditionalGeneration']


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [2]:
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor

# Pick a sample image from data/gemini_tie_pairs (or set SAMPLE_IMAGE_PATH manually).
image_candidates = sorted(Path("data/gemini_tie_pairs/red_blue/").glob("*.png")) + sorted(Path("data/gemini_tie_pairs").glob("*.jpg"))
if not image_candidates:
    raise FileNotFoundError("No images found under data/gemini_tie_pairs")

SAMPLE_IMAGE_PATH = image_candidates[0]
PROMPT = "Describe this image in 1-2 sentences."

image = Image.open(SAMPLE_IMAGE_PATH).convert("RGB")
processor = AutoProcessor.from_pretrained(MODEL_ID)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, images=image, return_tensors="pt")
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=96)

input_ids = inputs["input_ids"][0].tolist()
full_ids = output_ids[0].tolist()
new_ids = full_ids[len(input_ids):]

tok = getattr(processor, "tokenizer", None) or tokenizer
input_tokens = tok.convert_ids_to_tokens(input_ids)
output_tokens = tok.convert_ids_to_tokens(new_ids)

print(f"Image: {SAMPLE_IMAGE_PATH}")
print("\n=== INPUT (decoded) ===")
print(tok.decode(input_ids, skip_special_tokens=False))
print("\n=== INPUT TOKENS ===")
print(input_tokens)

print("\n=== OUTPUT (decoded) ===")
print(tok.decode(new_ids, skip_special_tokens=False))
print("\n=== OUTPUT TOKENS ===")
print(output_tokens)

Image: data/gemini_tie_pairs/red_blue/20260424_170804_pair_001_democrat_blue.png

=== INPUT (decoded) ===
<|im_start|>user
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|i

In [3]:
# Cleanup

del model
torch.cuda.empty_cache()

In [1]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "/project/jevans/tzhang3/models/gemma-4-31B-it"  # Or local path

dtype = "auto" if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
).eval()

print("Gemma4 loaded")
print("Class:", model.__class__.__name__)
print("Architectures:", getattr(model.config, "architectures", None))
print(model)

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Gemma4 loaded
Class: Gemma4ForConditionalGeneration
Architectures: ['Gemma4ForConditionalGeneration']
Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 5376, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=5376, out_features=4096, bias=False)
            (q_proj): Linear(in_features=5376, out_features=8192, bias=False)
            (v_proj): Linear(in_features=5376, out_features=4096, bias=False)
            (o_proj): Linear(in_features=8192, out_features=5376, bias=False)
            (head_out): Identity()
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=5376, out_features=21504, bias=False)
            (up_proj

In [ ]:
from pathlib import Path
from PIL import Image

# Pick a sample image from data/gemini_tie_pairs (or set SAMPLE_IMAGE_PATH manually).
image_candidates = sorted(Path("data/gemini_tie_pairs/red_blue/").glob("*.png")) + sorted(Path("data/gemini_tie_pairs").glob("*.jpg"))
if not image_candidates:
    raise FileNotFoundError("No images found under data/gemini_tie_pairs")

SAMPLE_IMAGE_PATH = image_candidates[0]
PROMPT = "Describe this image in 1-2 sentences."

image = Image.open(SAMPLE_IMAGE_PATH).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, images=image, return_tensors="pt")
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

# Gemma4 image grid metadata comes from preprocessed patch positions.
image_pos = inputs.get("image_position_ids")
if image_pos is None:
    raise KeyError(f"image_position_ids missing. Available keys: {list(inputs.keys())}")

# image_position_ids shape: (batch, max_patches, 2) with padding positions set to -1
pos0 = image_pos[0]
valid = (pos0[:, 0] >= 0) & (pos0[:, 1] >= 0)
if valid.any():
    patch_grid_w = int(pos0[valid, 0].max().item() + 1)
    patch_grid_h = int(pos0[valid, 1].max().item() + 1)
else:
    patch_grid_w = 0
    patch_grid_h = 0

pool_k = getattr(processor.image_processor, "pooling_kernel_size", 1)
grid_w = patch_grid_w // pool_k
grid_h = patch_grid_h // pool_k

num_soft_tokens = inputs.get("num_soft_tokens_per_image", [None])[0]

print(f"Image: {SAMPLE_IMAGE_PATH}")
print(f"Available input keys: {list(inputs.keys())}")
print(f"Patch grid (h, w): ({patch_grid_h}, {patch_grid_w})")
print(f"Token grid (h, w): ({grid_h}, {grid_w})")
print(f"num_soft_tokens_per_image[0]: {num_soft_tokens}")
if num_soft_tokens is not None:
    print(f"grid_h * grid_w: {grid_h * grid_w}")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=96)

input_ids = inputs["input_ids"][0].tolist()
full_ids = output_ids[0].tolist()
new_ids = full_ids[len(input_ids):]

tok = processor.tokenizer
input_tokens = tok.convert_ids_to_tokens(input_ids)
output_tokens = tok.convert_ids_to_tokens(new_ids)

print("\n=== INPUT (decoded) ===")
print(tok.decode(input_ids, skip_special_tokens=False))
print("\n=== INPUT TOKENS ===")
print(input_tokens)

print("\n=== OUTPUT (decoded) ===")
print(tok.decode(new_ids, skip_special_tokens=False))
print("\n=== OUTPUT TOKENS ===")
print(output_tokens)

Image: data/gemini_tie_pairs/red_blue/20260424_170804_pair_001_democrat_blue.png
Available input keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids']
Patch grid (h, w): (36, 66)
Token grid (h, w): (12, 22)
num_soft_tokens_per_image[0]: None

=== INPUT (decoded) ===
<bos><|turn>user
<|image><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|

: 

In [6]:
# Cleanup

del model
torch.cuda.empty_cache()

In [7]:
import torch
from transformers import AutoProcessor, MllamaForConditionalGeneration

MODEL_ID = "/project/jevans/tzhang3/models/Llama-3.2-11B-Vision-Instruct"  # Or meta-llama/Llama-3.2-11B-Vision-Instruct

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
).eval()

print("Llama 3.2 Vision loaded")
print("Class:", model.__class__.__name__)
print("Architectures:", getattr(model.config, "architectures", None))
print(model)

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

Llama 3.2 Vision loaded
Class: MllamaForConditionalGeneration
Architectures: ['MllamaForConditionalGeneration']
MllamaForConditionalGeneration(
  (model): MllamaModel(
    (vision_model): MllamaVisionModel(
      (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
      (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
        (tile_embedding): Embedding(9, 8197120)
      )
      (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
        (embedding): Embedding(9, 5120)
      )
      (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
        (embedding): Embedding(9, 5120)
      )
      (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (transformer): MllamaVisionEncoder(
        (layers): ModuleList(
          (0-31): 32 x MllamaVisionEncoderLayer(
            (self_attn): Ml

In [8]:
from pathlib import Path
from PIL import Image

# Pick a sample image from data/gemini_tie_pairs (or set SAMPLE_IMAGE_PATH manually).
image_candidates = sorted(Path("data/gemini_tie_pairs/red_blue/").glob("*.png")) + sorted(Path("data/gemini_tie_pairs").glob("*.jpg"))
if not image_candidates:
    raise FileNotFoundError("No images found under data/gemini_tie_pairs")

SAMPLE_IMAGE_PATH = image_candidates[0]
PROMPT = "Describe this image in 1-2 sentences."

image = Image.open(SAMPLE_IMAGE_PATH).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, images=image, return_tensors="pt")
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=96)

input_ids = inputs["input_ids"][0].tolist()
full_ids = output_ids[0].tolist()
new_ids = full_ids[len(input_ids):]

tok = processor.tokenizer
input_tokens = tok.convert_ids_to_tokens(input_ids)
output_tokens = tok.convert_ids_to_tokens(new_ids)

print(f"Image: {SAMPLE_IMAGE_PATH}")
print("\n=== INPUT (decoded) ===")
print(tok.decode(input_ids, skip_special_tokens=False))
print("\n=== INPUT TOKENS ===")
print(input_tokens)

print("\n=== OUTPUT (decoded) ===")
print(tok.decode(new_ids, skip_special_tokens=False))
print("\n=== OUTPUT TOKENS ===")
print(output_tokens)

Image: data/gemini_tie_pairs/red_blue/20260424_170804_pair_001_democrat_blue.png

=== INPUT (decoded) ===
<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|image|>Describe this image in 1-2 sentences.<|eot_id|><|start_header_id|>assistant<|end_header_id|>



=== INPUT TOKENS ===
['<|begin_of_text|>', '<|begin_of_text|>', '<|start_header_id|>', 'user', '<|end_header_id|>', 'ĊĊ', '<|image|>', 'Describe', 'Ġthis', 'Ġimage', 'Ġin', 'Ġ', '1', '-', '2', 'Ġsentences', '.', '<|eot_id|>', '<|start_header_id|>', 'assistant', '<|end_header_id|>', 'ĊĊ']

=== OUTPUT (decoded) ===
The image depicts a man in a suit standing in a hallway, with his hands clasped together. He has short, dark brown hair and is dressed in a blue suit, complemented by a white shirt and a blue tie. The background of the image is blurred, but it appears to be a hallway with white walls, marble columns, and a large arched doorway. The overall atmosphere suggests a formal setting, possibly a governm